In [13]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

BASE_DIR = 'C:/Users/SATWIK GHOSH/OneDrive/Desktop/AI-Powered Cyberthreat Analyzer'
PROCESSED_DIR = f'{BASE_DIR}/processed/'
MODELS_DIR = f'{BASE_DIR}/models/'
os.makedirs(MODELS_DIR, exist_ok=True)

print("--- Starting Consolidated Pipeline ---")

input_path = f'{PROCESSED_DIR}cicids_cleaned.parquet'
df = pd.read_parquet(input_path)
print(f"Step 1: Loaded Data. Initial shape: {df.shape}")

if 'label' in df.columns:
    le = LabelEncoder()
    df['target'] = le.fit_transform(df['label'])
    df = df.drop(columns=['label'])
    print(f"Step 2: Successfully converted text 'label' to numerical 'target'.")
    print(f"        Mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")
elif 'target' in df.columns:
    print("Step 2: 'target' already exists. Skipping encoding.")
else:
    print("CRITICAL ERROR: Neither 'label' nor 'target' found in dataset!")

corr_matrix = df.corr(numeric_only=True).abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

if 'target' in to_drop:
    to_drop.remove('target')

df_reduced = df.drop(columns=to_drop)
print(f"Step 3: Dropped {len(to_drop)} redundant features.")

try:
    X = df_reduced.drop(columns=['target'])
    y = df_reduced['target']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Step 4: Successfully split data!")
    print(f"        Training set (X_train): {X_train.shape}")
    print(f"        Testing set (X_test):   {X_test.shape}")
except KeyError as e:
    print(f"\nERROR during split! Python says: {e}")
    print("Here are the columns actually inside your dataset right now:")
    print(df_reduced.columns.tolist())

--- Starting Consolidated Pipeline ---
Step 1: Loaded Data. Initial shape: (225711, 79)
Step 2: Successfully converted text 'label' to numerical 'target'.
        Mapping: {'BENIGN': 0, 'DDoS': 1}
Step 3: Dropped 26 redundant features.
Step 4: Successfully split data!
        Training set (X_train): (180568, 52)
        Testing set (X_test):   (45143, 52)


In [14]:
from sklearn.preprocessing import StandardScaler
import joblib

print("\nScaling features using StandardScaler...")
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

joblib.dump(scaler, f'{MODELS_DIR}cicids_scaler.pkl')

X_train_scaled.to_parquet(f'{PROCESSED_DIR}X_train.parquet', engine='pyarrow')
X_test_scaled.to_parquet(f'{PROCESSED_DIR}X_test.parquet', engine='pyarrow')
pd.DataFrame(y_train).to_parquet(f'{PROCESSED_DIR}y_train.parquet', engine='pyarrow')
pd.DataFrame(y_test).to_parquet(f'{PROCESSED_DIR}y_test.parquet', engine='pyarrow')

print("Feature Engineering Complete! Scaled data saved.")


Scaling features using StandardScaler...
Feature Engineering Complete! Scaled data saved.
